# Data Understanding

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from utils import parse_number
from config import (
  RAW_REGENCIES_CSV,
  RAW_PROVINCES_CSV,
  UNDERSTANDING_PROVINCES_CSV,
  UNDERSTANDING_REGENCIES_CSV,
  PROVINCE_COLUMN_MAPPING,
  REGENCY_COLUMN_MAPPING
)

In [ ]:
# Memuat dataset hasil scraping mentah
df_prov = pd.read_csv(RAW_PROVINCES_CSV, dtype=str)
df_reg = pd.read_csv(RAW_REGENCIES_CSV, dtype=str)

# Pemetaan nama kolom ke format standar
df_prov = df_prov.rename(columns=PROVINCE_COLUMN_MAPPING)
df_reg = df_reg.rename(columns=REGENCY_COLUMN_MAPPING)

# Formatting dan casting tipe data numerik
PROVINCES_IGNORED_FORMATTED_COLS = ('no', 'province_name')
REGENCIES_IGNORED_FORMATTED_COLS = ('province_id', 'regency_no', 'regency_name')

for col in df_prov.columns:
  if col not in PROVINCES_IGNORED_FORMATTED_COLS:
    df_prov[col] = df_prov[col].apply(parse_number)

for col in df_reg.columns:
  if col not in REGENCIES_IGNORED_FORMATTED_COLS:
    df_reg[col] = df_reg[col].apply(parse_number)

numeric_cols = [c for c in df_reg.select_dtypes(include=[np.number]).columns if c not in REGENCIES_IGNORED_FORMATTED_COLS]

# Menyimpan hasil data yang telah dibersihkan dan diformat
os.makedirs(os.path.dirname(UNDERSTANDING_PROVINCES_CSV), exist_ok=True)
df_prov.to_csv(UNDERSTANDING_PROVINCES_CSV, index=False)
df_reg.to_csv(UNDERSTANDING_REGENCIES_CSV, index=False)

## Statistik Deskriptif

In [ ]:
print(df_reg[numeric_cols].describe(percentiles=[0.5]).T.round(2).to_markdown())

In [ ]:
print(f"Total Baris : {len(df_reg)}")
print(f"Total Kolom : {len(df_reg.columns)}")

## Eksplorasi Data

In [ ]:
top_provinces_limit = 10
top_regencies_limit = 10

### Heatmap Korelasi Pearson

In [ ]:
corr_matrix = df_reg[numeric_cols].corr().round(2)
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriks Korelasi Pearson Indikator KDMP')
plt.tight_layout()
plt.show()

### Barplot Top Provinsi Jumlah Koperasi

In [ ]:
top_prov = df_prov.sort_values(by='total_koperasi', ascending=True).tail(top_provinces_limit)
plt.figure(figsize=(10, 5))
sns.barplot(x='total_koperasi', y='province_name', data=top_prov, hue='province_name', legend=False, palette='Blues_r')
plt.title(f'{top_provinces_limit} Provinsi dengan Jumlah Koperasi Terbanyak di Indonesia')
plt.tight_layout()
plt.show()

### Barplot Top Kabupaten/Kota Nilai Transaksi

In [ ]:
top_reg = df_reg.sort_values(by='nilai_transaksi', ascending=True).tail(top_regencies_limit).copy()
top_reg['nilai_juta'] = top_reg['nilai_transaksi'] / 1e6
plt.figure(figsize=(10, 5))
sns.barplot(x='nilai_juta', y='regency_name', data=top_reg, hue='regency_name', legend=False, palette='viridis')
plt.title(f'{top_regencies_limit} Kabupaten/Kota dengan Nilai Transaksi Tertinggi (Juta Rp)')
plt.tight_layout()
plt.show()

### Distribusi Fitur

In [ ]:
num_features = len(numeric_cols)
ncols = 4
nrows = int(np.ceil(num_features / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()
colors = sns.color_palette('tab10', n_colors=num_features)

for idx, col in enumerate(numeric_cols):
  title = col.replace('_', ' ').title()
  use_log = df_reg[col].max() > 10000
  data = np.log1p(df_reg[col]) if use_log else df_reg[col]
  prefix = 'Log ' if use_log else ''
  sns.histplot(data, kde=True, ax=axes[idx], color=colors[idx])
  axes[idx].set_title(f'Distribusi {prefix}{title}')
  axes[idx].set_xlabel(f'{prefix}{title}')
  axes[idx].set_ylabel('Frekuensi')

for unused_ax in axes[num_features:]:
  fig.delaxes(unused_ax)

plt.tight_layout()
plt.show()

### Uji Relevansi & Seleksi Fitur

In [ ]:
# 1. Analisis Variabilitas & Koefisien Variasi (CV)
mean_val = df_reg[numeric_cols].mean()
std_val = df_reg[numeric_cols].std()
cv_val = (std_val / mean_val * 100).round(2)
var_val = df_reg[numeric_cols].var().round(2)

variability_df = pd.DataFrame({
  'Fitur': numeric_cols,
  'Mean': mean_val.round(2),
  'Std Dev': std_val.round(2),
  'Variance': var_val,
  'CV (%)': cv_val
})
print("=== 1. Uji Variabilitas Fitur ===")
print(variability_df.to_markdown(index=False))

# 2. Uji Multikolinearitas (Variance Inflation Factor - VIF)
X_scaled = StandardScaler().fit_transform(df_reg[numeric_cols].fillna(0))
cov_matrix = np.corrcoef(X_scaled, rowvar=False)
inv_corr = np.linalg.inv(cov_matrix)
vif_values = np.diag(inv_corr).round(2)

vif_df = pd.DataFrame({
  'Fitur': numeric_cols,
  'VIF Score': vif_values,
  'Kategori Multikolinearitas': ['Tinggi (>10)' if v > 10 else ('Sedang (5-10)' if v > 5 else 'Rendah (<5)') for v in vif_values]
})
print("\n=== 2. Uji Multikolinearitas (VIF) ===")
print(vif_df.to_markdown(index=False))

# 3. Analisis Komponen Utama (PCA Cumulative Variance)
pca = PCA().fit(X_scaled)
cum_var = np.cumsum(pca.explained_variance_ratio_ * 100).round(2)
pca_df = pd.DataFrame({
  'Komponen': [f'PC{i+1}' for i in range(len(numeric_cols))],
  'Explained Variance (%)': (pca.explained_variance_ratio_ * 100).round(2),
  'Cumulative Variance (%)': cum_var
})
print("\n=== 3. Analisis Komponen Utama (PCA Explained Variance) ===")
print(pca_df.to_markdown(index=False))

# Visualisasi Uji Seleksi Fitur
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x='VIF Score', y='Fitur', data=vif_df, ax=axes[0], palette='Blues_r', hue='Fitur', legend=False)
axes[0].axvline(10, color='red', linestyle='--', label='Ambang Batas Kritis (VIF=10)')
axes[0].set_title('Uji Multikolinearitas Fitur (VIF)')
axes[0].legend()

axes[1].plot(range(1, len(numeric_cols) + 1), cum_var, marker='o', color='teal', linewidth=2)
axes[1].axhline(80, color='orange', linestyle='--', label='Ambang Batas Informasi (80%)')
axes[1].set_title('Scree Plot / PCA Cumulative Explained Variance')
axes[1].set_xlabel('Jumlah Komponen Utama')
axes[1].set_ylabel('Kumulatif Varians (%)')
axes[1].set_ylim(0, 105)
axes[1].legend()

plt.tight_layout()
plt.show()

### Metrik Lainnya

In [ ]:
total_koperasi = int(df_prov['total_koperasi'].sum())
nib_sum = int(df_prov['koperasi_nib'].sum())
npwp_sum = int(df_prov['koperasi_npwp'].sum())
rat_sum = int(df_prov['koperasi_rat'].sum())
pct_nib = round(nib_sum / total_koperasi * 100, 2) if total_koperasi > 0 else 0.0
pct_npwp = round(npwp_sum / total_koperasi * 100, 2) if total_koperasi > 0 else 0.0
pct_rat = round(rat_sum / total_koperasi * 100, 2) if total_koperasi > 0 else 0.0
simpanan_pokok = float(df_prov['simpanan_pokok'].sum())
simpanan_wajib = float(df_prov['simpanan_wajib'].sum())
nilai_transaksi = float(df_prov['nilai_transaksi'].sum())

print(f"Total Kabupaten/Kota       : {len(df_reg)}")
print(f"Total Koperasi Terdata     : {total_koperasi:,} unit")
print(f"Koperasi Memiliki NIB      : {nib_sum:,} ({pct_nib}%)")
print(f"Koperasi Memiliki NPWP     : {npwp_sum:,} ({pct_npwp}%)")
print(f"Koperasi Telah RAT         : {rat_sum:,} ({pct_rat}%)")
print(f"Akumulasi Simpanan Pokok   : Rp {simpanan_pokok:,.2f}")
print(f"Akumulasi Simpanan Wajib   : Rp {simpanan_wajib:,.2f}")
print(f"Total Nilai Transaksi      : Rp {nilai_transaksi:,.2f}")

## Verifikasi Kualitas Data

### Mengecek Nilai Hilang

In [ ]:
print(pd.DataFrame({'Jumlah Null': df_reg.isnull().sum()}).to_markdown())

### Mengecek Outlier

In [ ]:
IGNORED_METADATA_COLUMNS = [
  'cluster_label',
  'regency_name',
  'province_name',
  'no',
  'regency_no',
  'province_id',
  'Province_ID',
  'No',
  'Kabupaten/Kota',
  'latitude',
  'longitude'
]

cols = [c for c in df_reg.columns if c not in IGNORED_METADATA_COLUMNS]
df_num = df_reg[cols].apply(pd.to_numeric, errors='coerce').fillna(0)

q1 = df_num.quantile(0.25)
q3 = df_num.quantile(0.75)
iqr = q3 - q1
lower = (q1 - 1.5 * iqr).clip(lower=0)
upper = q3 + 1.5 * iqr

out_cnt = ((df_num < lower) | (df_num > upper)).sum()
out_pct = (out_cnt / len(df_num) * 100).round(2)
skew_val = df_num.skew().round(2)

outliers_df = pd.DataFrame({
  'Nama Fitur': cols,
  'Skewness': skew_val.values,
  'Batas Bawah': lower.round(2).values,
  'Batas Atas': upper.round(2).values,
  'Outliers': out_cnt.values,
  'Persentase (%)': out_pct.values
})

print(outliers_df.to_markdown(index=False))